# Fine-tuning ViT (Google Pre-trained) on Higher Resolution Images

## Task: Binary Classification (Fake vs Real Faces) from Kaggle

**Paper Insight:** "When fine-tuning, we tend to use higher resolution images than during pre-training. This is beneficial... However, keeping the patch size the same results in a longer effective sequence length. Vision Transformers can handle arbitrary sequence lengths (up to memory constraints), but **positional embeddings need to be interpolated**." (Dosovitskiy et al., 2021)

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F

from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
import os
import time
import math

!pip install wandb -q
import wandb


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DL/'
    DATA_PATH = './data' # Download locally in colab initially
    print("✅ Connected to Google Drive")
except ImportError:
    print("❌ Not running on Colab, using local path")
    DATA_PATH = "./data"
    SAVE_PATH = "./"
    os.makedirs(SAVE_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Log in to W&B (Will ask for API Key on first run)
# wandb.login()

❌ Not running on Colab, using local path
Using device: cpu


## 1. Hyperparamters

In [3]:
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4
IMG_SIZE = 256

## 2. Dataset Preparation

In [9]:
!pip install kagglehub -q
import kagglehub

os.environ["KAGGLEHUB_CACHE"] = os.path.abspath(DATA_PATH)

path = kagglehub.dataset_download("ayushmandatta1/deepdetect-2025")

print("✅ Downloaded the dataset")

train_dir = os.path.join(path, "ddata", "train")
test_dir = os.path.join(path, "ddata", "test")

print(f"train_dir: {train_dir}")
print(f"test_dir: {test_dir}")


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ Downloaded the dataset
train_dir: /Users/duongtran/Documents/Projects/DLO/1_vision_transformer/data/datasets/ayushmandatta1/deepdetect-2025/versions/1/ddata/train
test_dir: /Users/duongtran/Documents/Projects/DLO/1_vision_transformer/data/datasets/ayushmandatta1/deepdetect-2025/versions/1/ddata/test


In [10]:
stats = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.TrivialAugmentWide(),
    transforms.RandomCrop(256, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [11]:
from torch.utils.data import random_split

full_train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

In [12]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=4, pin_memory=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)

## 3. Load & Interpolate Positional Embeddings

Since the resolution changed from 224 to 256, the number of patches changed:

- **Old:** `196`
- **New:** `256`

The 14x14 positional grid need to be interpolated to 16x16

In [ ]:
def load_state_dict_with_interpolation(model, state_dict):

    # 1. Get the current model's pos_embedding shape
    # (1, N+1, embed_dim) -> (1, 257, 768)
    new_pos_embed = model.encoder.pos_embedding.data
    num_patches_new = new_pos_embed.shape[1] - 1 #256

    # 2. Get loaded checkpoint's pos_embedding
    old_pos_embed = state_dict['encoder.pos_embedding'] # (1, 197, 768)

    # if shapes match, return imediately
    if old_pos_embed.shape == new_pos_embed.shape:
        print("Shapes match, no interpolation needed.")
        return state_dict

    # 3. Separate CLS token and Grid tokens
    old_cls_tokens = old_pos_embed[:, 0:1, :]
    old_grid_tokens = old_pos_embed[:, 1:, :]

    # 4. Reshape grid tokens to 2D image-like grid (B, C, H, W) for interpolation
    grid_size_old = int(math.sqrt(old_grid_tokens.shape[1]))
    embed_dim = old_grid_tokens.shape[-1]

    # Permute to (B, N, C) -> (B, C, N) -> (B, C, H, W) (1, 768, 14, 14)
    old_grid_tokens = old_grid_tokens.permute(0, 2, 1).reshape(1, embed_dim, grid_size_old, grid_size_old)

    # 5. Calculate new grid size
    grid_size_new = int(math.sqrt(num_patches_new))

    # 6. Interpolate
    new_grid_tokens = F.interpolate(
        old_cls_tokens,
        size=(grid_size_new, grid_size_new),
        mode='bicubic',
        align_corners=False
    )

    # 7. Reshape back to sequence (1, 256, 768) and permute back
    new_grid_tokens = new_grid_tokens.flatten(2).transpose(1, 2)

    # 8. Concatenate cls token back
    new_pos_embed = torch.cat((old_cls_tokens, new_grid_tokens), dim=1)

    # 9. Update state_dict
    state_dict['encoder.pos_embedding'] = new_pos_embed

    return state_dict

In [ ]:
# If MacOS activate this code

# import ssl
# ssl._create_default_https_context = ssl._create_unverified_context

In [15]:
# load standard ViT-Base-16 capable of processing IMG_SIZE
# but no loading weights yet
model = models.vit_b_16(weights=None, image_size=IMG_SIZE)

# Load standard ImageNet weights manually
pretrained_weights = models.ViT_B_16_Weights.IMAGENET1K_V1.get_state_dict(progress=True)

# Interpolate positional embeddings
pretrained_weights = load_state_dict_with_interpolation(model, pretrained_weights)

# Load the modified weights into the new model
model.load_state_dict(pretrained_weights)

# Modify Head for 2 Classes (Fake vs Real)
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, 2)

# Zero Initialization like in the ViT Paper
nn.init.zeros_(new_head.weight)
nn.init.zeros_(new_head.bias)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /Users/duongtran/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)>

## 4. Training

In [16]:
wandb.init(
    project="FT-ViT-Google-pretrained",
    config={
        "learning_rate": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE,
        "architecture": "ViT-B-16"
    }
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:


KeyboardInterrupt: 

In [ ]:
import numpy as np

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path for the checkpoint to be saved to.
                            Default: 'checkpoint.pth'
            trace_func (function): trace print function.
                            Default: print            
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
    
    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)

        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [20]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

/var/folders/qf/d2_g3wjs5yxddpfqpfp4j6hw0000gn/T/ipykernel_89591/1800294656.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
def train(model, loader, criterion, optimizer, scheduler):
    model.train()
    train_loss, train_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * x.size(0)
        train_acc += (out.argmax(1) == 1).sum().item()

    return train_loss / len(loader.dataset), train_acc / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    val_loss, val_acc = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            train_loss += loss.item() * x.size(0)
            val_acc += (out.argmax(1) == 1).sum().item()

    return train_loss / len(loader.dataset), val_acc / len(loader.dataset)

In [ ]:
def evaluate(model, loader):
    model.eval()
    eval_acc = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)

            eval_acc += (out.argmax(1) == 1).sum().item()

    return eval_acc / len(loader.dataset)

In [ ]:
train_accuracies, test_accuracies = [], []
early_stopping = EarlyStopping(patience=5, path='FT_GG_best_model.pth')

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scheduler)
    val_loss, val_acc = validate(model, loader, criterion)
    test_acc = evaluate(model, loader)

    scheduler.step()

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "lr": optimizer.param_groups[0]['lr']
    })

    print(f"{epoch+1}/{EPOCHS}: train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}," +
                             f" train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}, test_acc: {test_acc:.4f}")

    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("Early Stopping")
        break

wandb.finish()

In [ ]:
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Training and Test Accuracy")
plt.show()